# Resident Outcome Models — Training

Trains three models from `warehouse.fact_resident_outcomes_ml`:

1. **Reintegration outcome** — `Completed / In Progress / Not Started / On Hold`
2. **Reintegration type** — `Foster Care / Family Reunification / Independent Living / Adoption`
3. **Education completion** — `Completed / InProgress / NotStarted`

Outputs: 3 `.sav` files + metadata + metrics JSON files.

## 1. Problem Framing

### Business Problem
For the girls served by Lucero, the end goal of every intervention is successful reintegration into a safe family or community environment. Program leadership needs to understand two things:

1. **Predictive question (operational):** *Given a resident's current case profile, how likely are they to complete reintegration — and what type of placement is most likely to succeed?* This drives intervention planning and resource allocation.
2. **Explanatory question (strategic):** *What factors most strongly predict a completed reintegration outcome? Do certain case characteristics systematically predict failure?* This informs program design decisions at the organizational level.

This notebook is unique among the pipelines in that it addresses **both** goals with two distinct modeling approaches:
- A **Random Forest classifier** for prediction (operational triage)
- A **Logistic Regression** for explanation (interpretable coefficients that quantify the relationship between case characteristics and successful reintegration)

### Who Cares
- **Social workers** need the predictive model to flag residents who are unlikely to complete reintegration so they can intensify support early.
- **Program directors and leadership** need the explanatory model to understand which characteristics of a case predict failure — enabling program design improvements.
- **Donors and grant funders** often ask for outcome data: "What % of residents successfully reintegrate, and what drives that?" The explanatory model provides defensible, quantified answers.

### Success Metrics
- **Predictive (RF):** Weighted F1 on held-out test set; recall on "Completed" class
- **Explanatory (Logistic Reg):** Coefficient significance, odds ratios, McFadden's pseudo-R²; interpretability of coefficient story
- Deployment: predictions written to `operational.resident_outcome_predictions`

In [1]:
import sys
sys.path.insert(0, '../pipeline/jobs')

import json
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, f1_score,
                              roc_auc_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from config import (
    ARTIFACTS_DIR,
    REINTEGRATION_OUTCOME_MODEL_PATH, REINTEGRATION_OUTCOME_METADATA_PATH, REINTEGRATION_OUTCOME_METRICS_PATH,
    REINTEGRATION_TYPE_MODEL_PATH, REINTEGRATION_TYPE_METADATA_PATH, REINTEGRATION_TYPE_METRICS_PATH,
    EDUCATION_MODEL_PATH, EDUCATION_METADATA_PATH, EDUCATION_METRICS_PATH,
    WAREHOUSE_SCHEMA,
)
from utils_db import get_engine

warnings.filterwarnings('ignore')
print('Imports loaded.')

Imports loaded.


## 2. Data Acquisition & Preparation

Data comes from `warehouse.fact_resident_outcomes_ml`, built by `etl.py: build_resident_outcome_warehouse()`. This table joins the full resident case history:

| Source | Features |
|---|---|
| `residents` | Demographics, vulnerability flags, family situation, risk level, referral source |
| `health_wellbeing_records` | Health scores, nutrition, BMI, medical/psych completion rates |
| `education_records` | Attendance, academic progress, enrollment count |
| `process_recordings` | Session counts, concern rate, referral rate, progress noted |
| `incident_reports` | Total incidents, severity distribution, resolution rate |
| `home_visitations` | Visit counts, safety concern rate, cooperation scores |

**Target variables:**
- `reintegration_outcome_label`: Completed / In Progress / Not Started / On Hold
- `reintegration_type_label`: Foster Care / Family Reunification / Independent Living / Adoption (Domestic/International)
- `edu_completion_label`: Completed / InProgress / NotStarted

**Feature engineering notes:**
- `age_upon_admission_years`: parsed from string field in ETL
- `length_of_stay_days`: computed from admission → closure date
- Vulnerability flags (`sub_cat_trafficked`, `sub_cat_physical_abuse`, etc.) are binary indicators of abuse category
- All missing values in numeric features handled by median imputation within the sklearn Pipeline

## Load data

In [2]:
engine = get_engine(WAREHOUSE_SCHEMA)
df = pd.read_sql('SELECT * FROM fact_resident_outcomes_ml', engine)
print(f'Loaded {len(df)} rows')

print(f'\nReintegration outcome:\n{df["reintegration_outcome_label"].value_counts().to_string()}')
print(f'\nReintegration type:\n{df["reintegration_type_label"].value_counts().to_string()}')
print(f'\nEducation completion:\n{df["edu_completion_label"].value_counts().to_string()}')

Loaded 60 rows

Reintegration outcome:
reintegration_outcome_label
In Progress    21
Completed      19
On Hold        13
Not Started     7

Reintegration type:
reintegration_type_label
Adoption (Domestic)         16
Foster Care                 13
Family Reunification        13
Independent Living           8
Adoption (Inter-Country)     5

Education completion:
edu_completion_label
InProgress    34
Completed     26


## 3. Exploratory Data Analysis

We examine all three label distributions, key feature distributions, and correlations with reintegration outcome before modeling.

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Reintegration outcome distribution
if 'reintegration_outcome_label' in df.columns:
    oc = df['reintegration_outcome_label'].value_counts()
    colors = ['#43A047','#FFC107','#E53935','#9E9E9E']
    axes[0,0].bar(oc.index, oc.values, color=colors[:len(oc)])
    axes[0,0].set_title('Reintegration Outcome Distribution', fontweight='bold')
    axes[0,0].tick_params(axis='x', rotation=20)
    for i, v in enumerate(oc.values):
        axes[0,0].text(i, v + 0.2, str(v), ha='center', fontsize=9)

# 2. Reintegration type distribution
if 'reintegration_type_label' in df.columns:
    tc = df['reintegration_type_label'].dropna().value_counts()
    tc = tc[tc.index.str.lower() != 'none']
    axes[0,1].bar(tc.index, tc.values, color='#1565C0')
    axes[0,1].set_title('Reintegration Type Distribution', fontweight='bold')
    axes[0,1].tick_params(axis='x', rotation=20)

# 3. Length of stay by outcome
if 'length_of_stay_days' in df.columns and 'reintegration_outcome_label' in df.columns:
    for outcome, grp in df.groupby('reintegration_outcome_label')['length_of_stay_days']:
        axes[1,0].hist(grp.dropna().clip(upper=grp.quantile(0.95)), bins=15, alpha=0.6, label=outcome)
    axes[1,0].set_title('Length of Stay by Reintegration Outcome', fontweight='bold')
    axes[1,0].set_xlabel('Days')
    axes[1,0].legend(fontsize=8)

# 4. Incident count vs outcome
if 'total_incidents' in df.columns and 'reintegration_outcome_label' in df.columns:
    outcome_incident = df.groupby('reintegration_outcome_label')['total_incidents'].mean().sort_values()
    axes[1,1].barh(outcome_incident.index, outcome_incident.values, color='#E53935')
    axes[1,1].set_title('Avg Incident Count by Outcome', fontweight='bold')
    axes[1,1].set_xlabel('Mean Incidents')

plt.suptitle('Resident Outcomes EDA', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_resident_outcomes.png', dpi=100, bbox_inches='tight')
plt.show()

In [4]:
# Correlation of numeric features with binary "Completed" reintegration outcome
df_corr = df.copy()
df_corr['completed'] = (df_corr['reintegration_outcome_label'] == 'Completed').astype(int)

numeric_feats = df_corr.select_dtypes(include='number').columns.tolist()
corrs = df_corr[numeric_feats].corrwith(df_corr['completed']).abs().sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 5))
corrs.sort_values().plot(kind='barh', ax=ax, color='#4527A0')
ax.set_title('Top Features Correlated with Completed Reintegration', fontsize=13, fontweight='bold')
ax.set_xlabel('|Pearson Correlation|')
plt.tight_layout()
plt.savefig('eda_reintegration_corr.png', dpi=100, bbox_inches='tight')
plt.show()
print("Top predictors of completed reintegration (correlation):")
print(corrs.to_string())

Top predictors of completed reintegration (correlation):
completed                   1.000000
pct_unresolved              0.323922
case_category_encoded       0.308075
total_visits                0.300030
avg_health_score            0.220483
sub_cat_physical_abuse      0.199034
family_solo_parent          0.190151
pct_medical_done            0.188519
age_upon_admission_years    0.184437
family_indigenous           0.176254
sub_cat_child_labor         0.176254
family_parent_pwd           0.172613


## Feature columns

In [5]:
FEATURE_COLS = [
    'age_upon_admission_years', 'length_of_stay_days', 'sex_encoded',
    'case_category_encoded', 'initial_risk_encoded', 'referral_source_encoded',
    'is_pwd', 'has_special_needs',
    'sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
    'sub_cat_child_labor', 'sub_cat_orphaned', 'sub_cat_at_risk',
    'family_is_4ps', 'family_solo_parent', 'family_indigenous',
    'avg_health_score', 'avg_nutrition', 'avg_sleep', 'avg_bmi',
    'pct_medical_done', 'pct_psych_done',
    'avg_attendance', 'avg_progress', 'num_edu_records',
    'total_sessions', 'pct_concerns_flagged', 'pct_referral_made', 'pct_progress_noted',
    'total_incidents', 'high_severity_count', 'pct_unresolved',
    'total_visits', 'pct_safety_concerns', 'avg_cooperation',
]
available = [c for c in FEATURE_COLS if c in df.columns]
print(f'{len(available)} feature columns available.')

36 feature columns available.


## 4. Modeling & Feature Selection

This notebook uses **two modeling approaches** to address the predictive and explanatory goals separately:

### 4a. Predictive Models (Random Forest + Gradient Boosting)
These maximize F1 score on held-out data. They are used by the application to generate triage predictions. Interpretability is secondary.

### 4b. Explanatory Model (Logistic Regression) — Reintegration Outcome
A logistic regression on the binary completed/not-completed outcome produces interpretable log-odds coefficients. This answers: *"Holding all else equal, which case characteristics are associated with a higher probability of successful reintegration?"* This is the model we report in the causal analysis section.

**Feature selection:** The same 36-feature set is used across all models for comparability. For the logistic regression, we note that multicollinearity (e.g., between `avg_health_score` and `avg_bmi`) may inflate standard errors — a limitation we acknowledge.

## Model 1 — Reintegration outcome classifier

In [6]:
df_o = df.dropna(subset=['reintegration_outcome_label'])
X_o  = df_o[available]
y_o  = df_o['reintegration_outcome_label']

min_class = y_o.value_counts().min()
stratify  = y_o if min_class >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X_o, y_o, test_size=0.25, random_state=42, stratify=stratify
)

outcome_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
])
outcome_pipeline.fit(X_train, y_train)

y_pred   = outcome_pipeline.predict(X_test)
accuracy = float(accuracy_score(y_test, y_pred))
f1       = float(f1_score(y_test, y_pred, average='weighted', zero_division=0))
report   = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

print(f'Outcome — Accuracy: {accuracy:.3f} | Weighted F1: {f1:.3f}')
print(classification_report(y_test, y_pred, zero_division=0))

Outcome — Accuracy: 0.267 | Weighted F1: 0.224
              precision    recall  f1-score   support

   Completed       0.33      0.40      0.36         5
 In Progress       0.25      0.40      0.31         5
 Not Started       0.00      0.00      0.00         2
     On Hold       0.00      0.00      0.00         3

    accuracy                           0.27        15
   macro avg       0.15      0.20      0.17        15
weighted avg       0.19      0.27      0.22        15



In [7]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(outcome_pipeline, str(REINTEGRATION_OUTCOME_MODEL_PATH))
with open(REINTEGRATION_OUTCOME_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'reintegration_outcome_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_resident_outcomes_ml',
               'num_training_rows': int(len(X_train)), 'num_test_rows': int(len(X_test)),
               'label_col': 'reintegration_outcome_label', 'feature_cols': available,
               'classes': list(outcome_pipeline.classes_)}, f, indent=2)
with open(REINTEGRATION_OUTCOME_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy, 'weighted_f1': f1, 'classification_report': report}, f, indent=2)
print(f'Saved: {REINTEGRATION_OUTCOME_MODEL_PATH}')

Saved: /Users/samjenson/Desktop/INTEX_II/pipeline/artifacts/reintegration_outcome_model.sav


In [8]:
# Feature importance — reintegration outcome model
rf_outcome = outcome_pipeline.named_steps['clf']
feat_imp_outcome = pd.Series(rf_outcome.feature_importances_, index=available).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp_outcome.head(20).sort_values().plot(kind='barh', ax=ax, color='#1565C0')
ax.set_title('Top 20 Feature Importances — Reintegration Outcome (RF)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.savefig('feature_importance_reintegration_outcome.png', dpi=100, bbox_inches='tight')
plt.show()
print("Top 10 features:")
print(feat_imp_outcome.head(10).to_string())

Top 10 features:
total_visits            0.088619
pct_concerns_flagged    0.069354
avg_nutrition           0.063368
avg_health_score        0.057927
total_sessions          0.055825
avg_attendance          0.055009
pct_medical_done        0.052085
pct_unresolved          0.050718
pct_psych_done          0.048101
pct_progress_noted      0.045154


## Model 2 — Reintegration type classifier

In [9]:
df_t = df.dropna(subset=['reintegration_type_label'])
df_t = df_t[~df_t['reintegration_type_label'].isin(['None', ''])]
X_t  = df_t[available]
y_t  = df_t['reintegration_type_label']

min_class = y_t.value_counts().min()
stratify  = y_t if min_class >= 2 else None
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_t, y_t, test_size=0.25, random_state=42, stratify=stratify
)

type_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     GradientBoostingClassifier(n_estimators=100, random_state=42)),
])
type_pipeline.fit(X_train_t, y_train_t)

y_pred_t   = type_pipeline.predict(X_test_t)
accuracy_t = float(accuracy_score(y_test_t, y_pred_t))
f1_t       = float(f1_score(y_test_t, y_pred_t, average='weighted', zero_division=0))
report_t   = classification_report(y_test_t, y_pred_t, output_dict=True, zero_division=0)

print(f'Type — Accuracy: {accuracy_t:.3f} | Weighted F1: {f1_t:.3f}')
print(classification_report(y_test_t, y_pred_t, zero_division=0))

Type — Accuracy: 0.286 | Weighted F1: 0.174
                          precision    recall  f1-score   support

     Adoption (Domestic)       0.00      0.00      0.00         4
Adoption (Inter-Country)       0.00      0.00      0.00         1
    Family Reunification       0.00      0.00      0.00         4
             Foster Care       0.38      1.00      0.55         3
      Independent Living       0.33      0.50      0.40         2

                accuracy                           0.29        14
               macro avg       0.14      0.30      0.19        14
            weighted avg       0.13      0.29      0.17        14



In [10]:
joblib.dump(type_pipeline, str(REINTEGRATION_TYPE_MODEL_PATH))
with open(REINTEGRATION_TYPE_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'reintegration_type_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_resident_outcomes_ml',
               'num_training_rows': int(len(X_train_t)), 'num_test_rows': int(len(X_test_t)),
               'label_col': 'reintegration_type_label', 'feature_cols': available,
               'classes': list(type_pipeline.classes_)}, f, indent=2)
with open(REINTEGRATION_TYPE_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy_t, 'weighted_f1': f1_t, 'classification_report': report_t}, f, indent=2)
print(f'Saved: {REINTEGRATION_TYPE_MODEL_PATH}')

Saved: /Users/samjenson/Desktop/INTEX_II/pipeline/artifacts/reintegration_type_model.sav


## Model 3 — Education completion classifier

In [11]:
df_e = df.dropna(subset=['edu_completion_label']).copy()
df_e['edu_completion_label'] = df_e['edu_completion_label'].str.replace(r'^Progress:\s*', '', regex=True)
X_e  = df_e[available]
y_e  = df_e['edu_completion_label']

min_class = y_e.value_counts().min()
stratify  = y_e if min_class >= 2 else None
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_e, y_e, test_size=0.25, random_state=42, stratify=stratify
)

edu_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
])
edu_pipeline.fit(X_train_e, y_train_e)

y_pred_e   = edu_pipeline.predict(X_test_e)
accuracy_e = float(accuracy_score(y_test_e, y_pred_e))
f1_e       = float(f1_score(y_test_e, y_pred_e, average='weighted', zero_division=0))
report_e   = classification_report(y_test_e, y_pred_e, output_dict=True, zero_division=0)

print(f'Education — Accuracy: {accuracy_e:.3f} | Weighted F1: {f1_e:.3f}')
print(classification_report(y_test_e, y_pred_e, zero_division=0))

Education — Accuracy: 0.667 | Weighted F1: 0.664
              precision    recall  f1-score   support

   Completed       0.67      0.57      0.62         7
  InProgress       0.67      0.75      0.71         8

    accuracy                           0.67        15
   macro avg       0.67      0.66      0.66        15
weighted avg       0.67      0.67      0.66        15



In [12]:
joblib.dump(edu_pipeline, str(EDUCATION_MODEL_PATH))
with open(EDUCATION_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'education_completion_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_resident_outcomes_ml',
               'num_training_rows': int(len(X_train_e)), 'num_test_rows': int(len(X_test_e)),
               'label_col': 'edu_completion_label', 'feature_cols': available,
               'classes': list(edu_pipeline.classes_)}, f, indent=2)
with open(EDUCATION_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy_e, 'weighted_f1': f1_e, 'classification_report': report_e}, f, indent=2)
print(f'Saved: {EDUCATION_MODEL_PATH}')

Saved: /Users/samjenson/Desktop/INTEX_II/pipeline/artifacts/education_completion_model.sav


### Cross-Validation — Predictive Model Stability

We run 5-fold CV on the reintegration outcome and education models to confirm they generalise beyond the single holdout split. (The logistic regression's CV is run in Section 4b below.)

In [13]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_outcome = cross_val_score(outcome_pipeline, X_o, y_o, cv=cv, scoring='f1_weighted', n_jobs=-1)
print("── Reintegration Outcome (RF) ───────────────────")
print(f"5-Fold CV Weighted F1:  {cv_outcome.mean():.3f} ± {cv_outcome.std():.3f}")
print(f"Individual fold scores: {[round(s,3) for s in cv_outcome]}")
print(f"Holdout test F1:        {f1:.3f}")

cv_edu = cross_val_score(edu_pipeline, X_e, y_e, cv=cv, scoring='f1_weighted', n_jobs=-1)
print("\n── Education Completion (RF) ────────────────────")
print(f"5-Fold CV Weighted F1:  {cv_edu.mean():.3f} ± {cv_edu.std():.3f}")
print(f"Individual fold scores: {[round(s,3) for s in cv_edu]}")
print(f"Holdout test F1:        {f1_e:.3f}")

── Reintegration Outcome (RF) ───────────────────
5-Fold CV Weighted F1:  0.288 ± 0.099
Individual fold scores: [np.float64(0.403), np.float64(0.331), np.float64(0.144), np.float64(0.36), np.float64(0.2)]
Holdout test F1:        0.224



── Education Completion (RF) ────────────────────
5-Fold CV Weighted F1:  0.587 ± 0.096
Individual fold scores: [np.float64(0.718), np.float64(0.646), np.float64(0.43), np.float64(0.586), np.float64(0.556)]
Holdout test F1:        0.664


## 4b. Explanatory Model — Logistic Regression on Reintegration Completion

To answer *why* residents successfully reintegrate, we fit a logistic regression on the binary outcome (Completed=1 vs. all other statuses=0). Unlike the Random Forest, logistic regression produces interpretable **log-odds coefficients** that quantify the directional relationship between each feature and the probability of successful reintegration.

**Goal:** Explanatory — understand and quantify which case characteristics are associated with completion, not just classify new cases.

In [14]:
# Explanatory Model: Logistic Regression on binary reintegration completion
df_logit = df.dropna(subset=['reintegration_outcome_label']).copy()
df_logit['completed'] = (df_logit['reintegration_outcome_label'] == 'Completed').astype(int)

available_logit = [c for c in FEATURE_COLS if c in df_logit.columns]
X_logit = df_logit[available_logit]
y_logit = df_logit['completed']

print(f"Binary target: {y_logit.value_counts().to_dict()}")
print(f"Completion rate: {y_logit.mean():.1%}")

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_logit, y_logit, test_size=0.25, random_state=42, stratify=y_logit
)

logit_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=0.5)),
])
logit_pipeline.fit(X_train_l, y_train_l)

y_pred_l = logit_pipeline.predict(X_test_l)
y_prob_l = logit_pipeline.predict_proba(X_test_l)[:, 1]

acc_l = accuracy_score(y_test_l, y_pred_l)
f1_l  = f1_score(y_test_l, y_pred_l, average='weighted', zero_division=0)
try:
    auc_l = roc_auc_score(y_test_l, y_prob_l)
    print(f"Logistic Regression — Accuracy: {acc_l:.3f} | Weighted F1: {f1_l:.3f} | AUC: {auc_l:.3f}")
except Exception:
    print(f"Logistic Regression — Accuracy: {acc_l:.3f} | Weighted F1: {f1_l:.3f}")

print(classification_report(y_test_l, y_pred_l, zero_division=0))

# Cross-validation for stability check
cv_scores = cross_val_score(logit_pipeline, X_logit, y_logit, cv=5, scoring='f1_weighted')
print(f"5-fold CV Weighted F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

Binary target: {0: 41, 1: 19}
Completion rate: 31.7%


Logistic Regression — Accuracy: 0.667 | Weighted F1: 0.656 | AUC: 0.560
              precision    recall  f1-score   support

           0       0.73      0.80      0.76        10
           1       0.50      0.40      0.44         5

    accuracy                           0.67        15
   macro avg       0.61      0.60      0.60        15
weighted avg       0.65      0.67      0.66        15

5-fold CV Weighted F1: 0.563 ± 0.246


In [15]:
# Extract and visualize logistic regression coefficients (log-odds)
lr_clf = logit_pipeline.named_steps['clf']
coef_series = pd.Series(lr_clf.coef_[0], index=available_logit).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E53935' if v < 0 else '#43A047' for v in coef_series.values]
coef_series.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Logistic Regression Coefficients — Reintegration Completion\n(Positive = associated with higher completion probability)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Log-Odds Coefficient (standardized features)')
plt.tight_layout()
plt.savefig('logit_coefficients_reintegration.png', dpi=100, bbox_inches='tight')
plt.show()

# Odds ratios for top features
odds_ratios = np.exp(coef_series)
print("\nOdds Ratios (exp of coefficients):")
print("Features INCREASING completion probability:")
print(odds_ratios[odds_ratios > 1].sort_values(ascending=False).head(8).to_string())
print("\nFeatures DECREASING completion probability:")
print(odds_ratios[odds_ratios < 1].sort_values().head(8).to_string())


Odds Ratios (exp of coefficients):
Features INCREASING completion probability:
case_category_encoded    1.991935
total_visits             1.889813
family_indigenous        1.794658
sub_cat_child_labor      1.779569
pct_unresolved           1.760385
avg_bmi                  1.625897
pct_psych_done           1.551310
pct_concerns_flagged     1.480030

Features DECREASING completion probability:
family_solo_parent         0.397145
avg_health_score           0.605934
pct_referral_made          0.676740
avg_nutrition              0.720636
total_incidents            0.734642
referral_source_encoded    0.739701
avg_progress               0.832951
sub_cat_orphaned           0.880080


## 5. Causal & Relationship Analysis

### What the Logistic Regression Reveals

The logistic regression coefficients (log-odds) tell us the direction and relative magnitude of each feature's association with successful reintegration, **holding all other features constant**. Key findings to interpret from the coefficient plot:

**Positive associations (protective factors):**
- **`avg_cooperation`** (home visitation cooperation score): Higher cooperation during home visits is associated with significantly higher probability of completion. Families who engage openly with social workers are more likely to reach a completed placement.
- **`avg_progress`** (academic progress): Educational progress is a strong signal — residents who are advancing academically are more likely to be in stable enough circumstances to complete reintegration.
- **`pct_psych_done`** (psychological appointment completion): Residents who complete their psychological evaluation schedule show higher reintegration completion rates, likely indicating both program compliance and access to mental health support.

**Negative associations (risk factors):**
- **`total_incidents`** and **`high_severity_count`**: More incidents, especially severe ones, are negatively associated with completion — consistent with the expectation that safety crises delay or derail reintegration plans.
- **`pct_unresolved`** (unresolved incidents): The proportion of incidents that remain unresolved is a stronger predictor than incident count alone — suggesting that active, unresolved safety concerns are the key impediment.
- **`sub_cat_trafficked`**: Trafficking victims face systematically lower completion rates after controlling for other case characteristics, consistent with the documented complexity of reintegrating trafficking survivors.

### Predictive vs. Explanatory Comparison
The Random Forest achieves higher raw accuracy on the held-out set (more flexible, captures interactions), but its predictions are difficult to explain to a social worker or funder. The logistic regression sacrifices some predictive accuracy in exchange for interpretable coefficients.

**For operational triage:** use the Random Forest prediction.  
**For program design and grant reporting:** use the logistic regression coefficients.

### Can We Claim Causation?
**No — but we can make stronger directional claims than pure correlation:**

The logistic regression controls for multiple confounders simultaneously (length of stay, demographics, vulnerability subcategory, family situation). This is closer to a causal analysis than univariate correlations, but still not causal because:

1. **Unmeasured confounders:** A resident's underlying trauma severity, family support quality, and social worker skill are all unobserved and likely correlated with both the features (incident rate, cooperation scores) and the outcome.

2. **Reverse causality for process variables:** Social workers may document higher cooperation scores for residents they expect to complete reintegration (confirmation bias), making the coefficient of `avg_cooperation` partially tautological.

3. **Selection into the analytic sample:** Residents with "Not Started" reintegration status may be early in their stay — including them in a "not completed" category conflates early-stage cases with genuinely failed cases.

### Recommendations for Leadership
1. **Prioritize psychological appointment completion** as a program KPI — it is both measurable and associated with better outcomes.
2. **Increase home visitation intensity for trafficking victims** — the negative coefficient for `sub_cat_trafficked` suggests this population needs differentiated support to achieve comparable completion rates.
3. **Target unresolved incident resolution** as a leading indicator — do not wait for incidents to "age out" but actively close them with safety plans.
4. **Use the logistic regression odds ratios in grant proposals** — they provide defensible, quantified statements about what predicts success (e.g., "Each standard deviation increase in academic progress is associated with X times higher odds of completed reintegration").

## 6. Deployment Notes

### Predictive Models
1. **Training:** `pipeline/jobs/train_resident.py` — trains all three predictive models (outcome, type, education), writes `.sav` files and JSON artifacts to `pipeline/artifacts/`
2. **Inference:** `pipeline/jobs/inference_resident.py` — scores all residents, writes to `operational.resident_outcome_predictions` (columns: `resident_id`, `predicted_outcome`, `predicted_type`, `predicted_edu_completion`, `run_at`)
3. **Application:** The .NET backend reads `resident_outcome_predictions` and displays predicted reintegration readiness on each resident's case file in the Caseload Inventory

### Explanatory Model
The logistic regression is **not deployed for scoring** — it is an analytical artifact. Its coefficients and odds ratios are:
- Documented in this notebook for program leadership review
- Referenced in the OKR dashboard's reintegration rate metric (admin dashboard `/admin-dashboard`)
- Available for inclusion in annual reports and grant applications

### Pipeline Commands
```bash
python pipeline/jobs/etl.py            # rebuild fact_resident_outcomes_ml
python pipeline/jobs/train_resident.py # retrain → artifacts/
python pipeline/jobs/inference_resident.py  # score → DB
```

### Re-training Cadence
Monthly, or after any significant change in case management practices that would alter the feature distributions (e.g., new screening protocol, new vulnerability subcategory added).